In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.0 MB/s eta 0:00:00


In [4]:
# 1. Core libraries
import pandas as pd
import numpy as np

from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_STATE = 42

In [5]:

DATA_DIR = "/content/drive/MyDrive/Project/Capstone/Data/Processed"

trainsplit_df = pd.read_parquet(f"{DATA_DIR}/train_split.parquet")
valsplit_df   = pd.read_parquet(f"{DATA_DIR}/val_split.parquet")
eval_df       = pd.read_parquet(f"{DATA_DIR}/eval_holdout.parquet")

print("Train split :", trainsplit_df.shape)
print("Val split   :", valsplit_df.shape)
print("Eval holdout:", eval_df.shape)

trainsplit_df.head(3)

Train split : (3600000, 24)
Val split   : (900000, 24)
Eval holdout: (350000, 24)


,city_id,store_id,management_group_id,first_category_id,second_category_id,third_category_id,product_id,dt,sale_amount,hours_sale,...,activity_flag,precpt,avg_temperature,avg_humidity,avg_wind_level,stockout_hours,is_censored_day,is_not_stocked,is_partial_stockout,shelf_life_bucket
0,0,0,0,5,6,65,38,2024-03-28,0.1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, ...",...,0,1.6999,15.48,73.54,1.97,0,0,0,0,Short_Life
1,0,0,0,5,6,65,38,2024-03-29,0.1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, ...",...,0,3.0190,15.08,76.56,1.71,1,0,0,1,Short_Life
2,0,0,0,5,6,65,38,2024-03-30,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0,2.0942,15.91,76.47,1.73,0,0,1,0,Short_Life


In [6]:
target_col = "saleamount"

base_features = [
    "dayofweek", "isweekend", "month",
    "precpt", "avgtemperature", "avghumidity", "avgwindlevel",
    "discount", "holidayflag", "activityflag",
    "stockouthours",           # stockout intensity 06-22
    "iscensoredday",           # Type B censored demand
    "isnotstocked"             # Type A genuine zero demand
]

id_features = [
    "cityid", "storeid", "managementgroupid",
    "firstcategoryid", "secondcategoryid", "thirdcategoryid", "productid"
]

final_features = base_features + id_features

missing_cols = [c for c in final_features if c not in trainsplit_df.columns]
print("Missing in train:", missing_cols)

Missing in train: ['dayofweek', 'isweekend', 'month', 'avgtemperature', 'avghumidity', 'avgwindlevel', 'holidayflag', 'activityflag', 'stockouthours', 'iscensoredday', 'isnotstocked', 'cityid', 'storeid', 'managementgroupid', 'firstcategoryid', 'secondcategoryid', 'thirdcategoryid', 'productid']


In [7]:
target_col = "sale_amount"

base_features = [
    "dayofweek", "isweekend", "month",
    "precpt", "avg_temperature", "avg_humidity", "avg_wind_level",
    "discount", "holiday_flag", "activity_flag",
    "stockout_hours",          # stockout intensity 06-22
    "iscensoredday",           # Type B censored demand
    "isnotstocked",            # Type A genuine zero demand
]

id_features = [
    "city_id", "store_id", "management_group_id",
    "first_category_id", "second_category_id", "third_category_id", "product_id",
]

final_features = base_features + id_features

missing_cols = [c for c in final_features if c not in trainsplit_df.columns]
print("Missing in train:", missing_cols)

Missing in train: ['dayofweek', 'isweekend', 'month', 'iscensoredday', 'isnotstocked']


In [8]:

# 2a. Temporal features (intraday/weekly patterns for RQ-a)
for df in (trainsplit_df, valsplit_df, eval_df):
    df["dt"] = pd.to_datetime(df["dt"])
    df["dayofweek"] = df["dt"].dt.dayofweek
    df["isweekend"] = df["dayofweek"].isin([5, 6]).astype(int)
    df["month"] = df["dt"].dt.month

# 2b. Stockout-aware censoring (RQ-b, RQ-c)
for df in (trainsplit_df, valsplit_df, eval_df):
    # stockout_hours already exists and is your corrected count of stockout hours 6-22
    df["stockout_hours"] = df["stockout_hours"].astype(int)

    # TYPE A: never stocked (true zero demand)
    df["isnotstocked"] = (
        (df["stockout_hours"] == 0) & (df["sale_amount"] == 0)
    ).astype(int)

    # TYPE B: sold out mid-day (censored demand)
    df["iscensoredday"] = (df["stockout_hours"] >= 3).astype(int)

    # Optional: partial stockout 1–2 hours (not in base_features, but available)
    df["ispartialstockout"] = (
        (df["stockout_hours"] >= 1) & (df["stockout_hours"] < 3)
    ).astype(int)

# 2c. Shelf-life segmentation using third_category_id proxy
def assign_shelf_life(catid):
    if catid <= 80:
        return "ShortLife"
    elif catid <= 160:
        return "MediumLife"
    else:
        return "LongLife"

for df in (trainsplit_df, valsplit_df, eval_df):
    df["shelflifebucket"] = df["third_category_id"].apply(assign_shelf_life)

print("Feature engineering complete.")
print("Columns now include:", [c for c in trainsplit_df.columns if c in base_features + id_features])

Feature engineering complete.
Columns now include: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level', 'stockout_hours', 'dayofweek', 'isweekend', 'month', 'isnotstocked', 'iscensoredday']


In [9]:
missing_cols = [c for c in final_features if c not in trainsplit_df.columns]
print("Missing in train:", missing_cols)

Missing in train: []


In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd

RANDOM_STATE = 42

def wape(y_true, y_pred):
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(y_true - y_pred)) / denom * 100


def train_rf_for_segment(segment_name, train_df, val_df, features, target_col):
    """
    Train a Random Forest for one shelf-life segment.
    Returns (model, metrics dict).
    """
    train_seg = train_df[train_df["shelflifebucket"] == segment_name].copy()
    val_seg   = val_df[val_df["shelflifebucket"] == segment_name].copy()

    if train_seg.empty or val_seg.empty:
        print(f"⚠️ Segment {segment_name}: no data, skipping.")
        return None, None

    X_train = train_seg[features].fillna(0)
    y_train = train_seg[target_col].astype(float)

    X_val   = val_seg[features].fillna(0)
    y_val   = val_seg[target_col].astype(float)

    print(f"\n--- Training RF for segment: {segment_name} ---")
    print("Train rows:", len(X_train), "| Val rows:", len(X_val))

    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_split=20,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)

    seg_r2   = r2_score(y_val, y_pred)
    seg_wape = wape(y_val, y_pred)

    print(f"RF ({segment_name}) -> R2: {seg_r2:.4f} | WAPE: {seg_wape:.2f}%")

    metrics = {
        "Shelf Life Segment": segment_name,
        "Algorithm Strategy": "Random Forest Only",
        "Eval R2": seg_r2,
        "Eval WAPE": seg_wape,
    }

    return rf, metrics

In [11]:
segments = ["ShortLife", "MediumLife", "LongLife"]

rf_models = {}
rf_results = []

for seg in segments:
    model, metrics = train_rf_for_segment(
        seg,
        trainsplit_df,
        valsplit_df,
        final_features,
        target_col,
    )
    if model is not None:
        rf_models[seg] = model
        rf_results.append(metrics)

rf_results_df = pd.DataFrame(rf_results)

print("\n🎯 RANDOM FOREST SEGMENT RESULTS (Validation):")
display(
    rf_results_df.style.format(
        {"Eval R2": "{:.4f}", "Eval WAPE": "{:.2f}%"}
    )
)


--- Training RF for segment: ShortLife ---
Train rows: 940104 | Val rows: 235026
RF (ShortLife) -> R2: 0.6618 | WAPE: 38.95%

--- Training RF for segment: MediumLife ---
Train rows: 1708128 | Val rows: 427032
RF (MediumLife) -> R2: 0.4795 | WAPE: 42.26%

--- Training RF for segment: LongLife ---
Train rows: 951768 | Val rows: 237942
RF (LongLife) -> R2: 0.9082 | WAPE: 35.92%

🎯 RANDOM FOREST SEGMENT RESULTS (Validation):


,Shelf Life Segment,Algorithm Strategy,Eval R2,Eval WAPE
0,ShortLife,Random Forest Only,0.6618,38.95%
1,MediumLife,Random Forest Only,0.4795,42.26%
2,LongLife,Random Forest Only,0.9082,35.92%


In [14]:
import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation

def train_lgbm_for_segment(segment_name, train_df, val_df, features, target_col):
    """
    Train a LightGBM regressor for one shelf-life segment.
    Returns (model, metrics dict).
    """
    train_seg = train_df[train_df["shelflifebucket"] == segment_name].copy()
    val_seg   = val_df[val_df["shelflifebucket"] == segment_name].copy()

    if train_seg.empty or val_seg.empty:
        print(f"⚠️ Segment {segment_name}: no data, skipping.")
        return None, None

    X_train = train_seg[features].fillna(0)
    y_train = train_seg[target_col].astype(float)

    X_val   = val_seg[features].fillna(0)
    y_val   = val_seg[target_col].astype(float)

    print(f"\n--- Training LGBM for segment: {segment_name} ---")
    print("Train rows:", len(X_train), "| Val rows:", len(X_val))

    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_val   = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

    lgb_params = {
        "objective": "regression",
        "metric": "l2",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "min_data_in_leaf": 50,
        "verbose": -1,
        "seed": RANDOM_STATE,
    }

    callbacks = [
        early_stopping(stopping_rounds=50, verbose=False),
        log_evaluation(period=0),  # silence per-iteration logs
    ]

    model = lgb.train(
        lgb_params,
        lgb_train,
        num_boost_round=1000,
        valid_sets=[lgb_train, lgb_val],
        valid_names=["train", "val"],
        callbacks=callbacks,
    )

    y_pred = model.predict(X_val, num_iteration=model.best_iteration)

    seg_r2   = r2_score(y_val, y_pred)
    seg_wape = wape(y_val, y_pred)

    print(f"LGBM ({segment_name}) -> R2: {seg_r2:.4f} | WAPE: {seg_wape:.2f}%")

    metrics = {
        "Shelf Life Segment": segment_name,
        "Algorithm Strategy": "LightGBM Only",
        "Eval R2": seg_r2,
        "Eval WAPE": seg_wape,
    }

    return model, metrics

In [15]:
segments = ["ShortLife", "MediumLife", "LongLife"]

lgbm_models = {}
lgbm_results = []

for seg in segments:
    model, metrics = train_lgbm_for_segment(
        seg,
        trainsplit_df,
        valsplit_df,
        final_features,
        target_col,
    )
    if model is not None:
        lgbm_models[seg] = model
        lgbm_results.append(metrics)

lgbm_results_df = pd.DataFrame(lgbm_results)

print("\n🎯 LIGHTGBM SEGMENT RESULTS (Validation):")
display(
    lgbm_results_df.style.format(
        {"Eval R2": "{:.4f}", "Eval WAPE": "{:.2f}%"}
    )
)


--- Training LGBM for segment: ShortLife ---
Train rows: 940104 | Val rows: 235026
LGBM (ShortLife) -> R2: 0.7074 | WAPE: 36.18%

--- Training LGBM for segment: MediumLife ---
Train rows: 1708128 | Val rows: 427032
LGBM (MediumLife) -> R2: 0.5984 | WAPE: 37.80%

--- Training LGBM for segment: LongLife ---
Train rows: 951768 | Val rows: 237942
LGBM (LongLife) -> R2: 0.9228 | WAPE: 32.04%

🎯 LIGHTGBM SEGMENT RESULTS (Validation):


,Shelf Life Segment,Algorithm Strategy,Eval R2,Eval WAPE
0,ShortLife,LightGBM Only,0.7074,36.18%
1,MediumLife,LightGBM Only,0.5984,37.80%
2,LongLife,LightGBM Only,0.9228,32.04%


In [16]:
segments = ["ShortLife", "MediumLife", "LongLife"]

ensemble_results = []

for seg in segments:
    train_seg = trainsplit_df[trainsplit_df["shelflifebucket"] == seg].copy()
    val_seg   = valsplit_df[valsplit_df["shelflifebucket"] == seg].copy()

    if train_seg.empty or val_seg.empty:
        continue

    X_val = val_seg[final_features].fillna(0)
    y_val = val_seg[target_col].astype(float)

    rf = rf_models.get(seg)
    lgbm = lgbm_models.get(seg)

    if rf is None or lgbm is None:
        continue

    rf_pred  = rf.predict(X_val)
    lgb_pred = lgbm.predict(X_val, num_iteration=lgbm.best_iteration)

    # 50/50 ensemble
    ens_pred = 0.5 * rf_pred + 0.5 * lgb_pred
    ens_r2   = r2_score(y_val, ens_pred)
    ens_wape = wape(y_val, ens_pred)

    ensemble_results.append({
        "Shelf Life Segment": seg,
        "Algorithm Strategy": "Ensemble (50% RF + 50% LGBM)",
        "Eval R2": ens_r2,
        "Eval WAPE": ens_wape,
    })

ensemble_results_df = pd.DataFrame(ensemble_results)

print("\n🎯 RF + LGBM ENSEMBLE RESULTS (Validation):")
display(
    ensemble_results_df.style.format(
        {"Eval R2": "{:.4f}", "Eval WAPE": "{:.2f}%"}
    )
)


🎯 RF + LGBM ENSEMBLE RESULTS (Validation):


,Shelf Life Segment,Algorithm Strategy,Eval R2,Eval WAPE
0,ShortLife,Ensemble (50% RF + 50% LGBM),0.6960,36.64%
1,MediumLife,Ensemble (50% RF + 50% LGBM),0.5638,38.89%
2,LongLife,Ensemble (50% RF + 50% LGBM),0.9194,32.85%


In [17]:
# Merge all three tables on Shelf Life Segment
rf_df    = rf_results_df.set_index("Shelf Life Segment")
lgbm_df  = lgbm_results_df.set_index("Shelf Life Segment")
ens_df   = ensemble_results_df.set_index("Shelf Life Segment")

smart_rows = []

for seg in segments:
    if seg not in rf_df.index or seg not in lgbm_df.index or seg not in ens_df.index:
        continue

    # Choose strategy: start by picking the lowest WAPE
    candidates = {
        "Random Forest Only": (rf_df.loc[seg, "Eval R2"],   rf_df.loc[seg, "Eval WAPE"]),
        "LightGBM Only":      (lgbm_df.loc[seg, "Eval R2"], lgbm_df.loc[seg, "Eval WAPE"]),
        "Ensemble (50% RF + 50% LGBM)": (ens_df.loc[seg, "Eval R2"], ens_df.loc[seg, "Eval WAPE"]),
    }

    # Here we optimize primarily for WAPE (business error), then R2 as tie-breaker
    best_strategy = min(
        candidates.items(),
        key=lambda kv: (kv[1][1], -kv[1][0])  # (WAPE, -R2)
    )

    strategy_name, (best_r2, best_wape) = best_strategy

    smart_rows.append({
        "Shelf Life Segment": seg,
        "Algorithm Strategy": strategy_name,
        "Eval R2": best_r2,
        "Eval WAPE": best_wape,
    })

smart_df = pd.DataFrame(smart_rows)

print("\n🎯 THE SMART SEGMENT STRATEGY RESULTS (Validation):")
display(
    smart_df.style.format(
        {"Eval R2": "{:.4f}", "Eval WAPE": "{:.2f}%"}
    )
)


🎯 THE SMART SEGMENT STRATEGY RESULTS (Validation):


,Shelf Life Segment,Algorithm Strategy,Eval R2,Eval WAPE
0,ShortLife,LightGBM Only,0.7074,36.18%
1,MediumLife,LightGBM Only,0.5984,37.80%
2,LongLife,LightGBM Only,0.9228,32.04%


In [18]:
category_col = "first_category_id"  # top-level product category

cat_results = []

unique_cats = sorted(trainsplit_df[category_col].unique())

for cat in unique_cats:
    train_cat = trainsplit_df[trainsplit_df[category_col] == cat].copy()
    val_cat   = valsplit_df[valsplit_df[category_col] == cat].copy()

    # Skip tiny categories to avoid noisy segments
    if len(train_cat) < 5000 or len(val_cat) < 1000:
        continue

    X_train = train_cat[final_features].fillna(0)
    y_train = train_cat[target_col].astype(float)

    X_val   = val_cat[final_features].fillna(0)
    y_val   = val_cat[target_col].astype(float)

    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_val   = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

    params = {
        "objective": "regression",
        "metric": "l2",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "min_data_in_leaf": 50,
        "verbose": -1,
        "seed": RANDOM_STATE,
    }

    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=500,
        valid_sets=[lgb_train, lgb_val],
        valid_names=["train", "val"],
        callbacks=[
            early_stopping(stopping_rounds=50, verbose=False),
            log_evaluation(period=0),
        ],
    )

    y_pred = model.predict(X_val, num_iteration=model.best_iteration)

    r2_val   = r2_score(y_val, y_pred)
    wape_val = wape(y_val, y_pred)

    cat_results.append({
        "Category Level": "first_category_id",
        "Category ID": cat,
        "Train Samples": len(train_cat),
        "Eval R2": r2_val,
        "Eval WAPE": wape_val,
    })

cat_results_df = pd.DataFrame(cat_results)
print("\n📦 CATEGORY-ONLY LGBM RESULTS (first_category_id):")
display(
    cat_results_df.sort_values("Eval WAPE").style.format(
        {"Eval R2": "{:.4f}", "Eval WAPE": "{:.2f}%"}
    )
)


📦 CATEGORY-ONLY LGBM RESULTS (first_category_id):


,Category Level,Category ID,Train Samples,Eval R2,Eval WAPE
14,first_category_id,21,204696,0.9308,29.01%
16,first_category_id,23,21672,0.3631,31.22%
11,first_category_id,17,5184,0.2802,32.71%
23,first_category_id,31,137664,0.5328,33.29%
19,first_category_id,27,16560,0.3100,33.31%
18,first_category_id,25,29664,0.7789,33.50%
2,first_category_id,4,783936,0.7057,33.83%
10,first_category_id,16,516744,0.4276,33.92%
17,first_category_id,24,99576,0.6938,33.97%
15,first_category_id,22,23544,0.2021,34.30%


In [19]:
# Combined segment: shelf life + top-level category
for df in (trainsplit_df, valsplit_df):
    df["segment_cat"] = (
        df["shelflifebucket"].astype(str)
        + "_cat"
        + df["first_category_id"].astype(str)
    )

segcat_results = []

unique_segcats = sorted(trainsplit_df["segment_cat"].unique())

for sc in unique_segcats:
    train_sc = trainsplit_df[trainsplit_df["segment_cat"] == sc].copy()
    val_sc   = valsplit_df[valsplit_df["segment_cat"] == sc].copy()

    if len(train_sc) < 5000 or len(val_sc) < 1000:
        continue

    X_train = train_sc[final_features].fillna(0)
    y_train = train_sc[target_col].astype(float)

    X_val   = val_sc[final_features].fillna(0)
    y_val   = val_sc[target_col].astype(float)

    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_val   = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=500,
        valid_sets=[lgb_train, lgb_val],
        valid_names=["train", "val"],
        callbacks=[
            early_stopping(stopping_rounds=50, verbose=False),
            log_evaluation(period=0),
        ],
    )

    y_pred = model.predict(X_val, num_iteration=model.best_iteration)

    r2_val   = r2_score(y_val, y_pred)
    wape_val = wape(y_val, y_pred)

    segcat_results.append({
        "Segment+Category": sc,
        "Train Samples": len(train_sc),
        "Eval R2": r2_val,
        "Eval WAPE": wape_val,
    })

segcat_results_df = pd.DataFrame(segcat_results)
print("\n📦 SHELF-LIFE + CATEGORY LGBM RESULTS:")
display(
    segcat_results_df.sort_values("Eval WAPE").style.format(
        {"Eval R2": "{:.4f}", "Eval WAPE": "{:.2f}%"}
    )
)


📦 SHELF-LIFE + CATEGORY LGBM RESULTS:


,Segment+Category,Train Samples,Eval R2,Eval WAPE
5,LongLife_cat21,62928,0.9316,26.05%
21,MediumLife_cat28,79488,0.5025,30.51%
34,ShortLife_cat24,19800,0.6673,30.84%
27,ShortLife_cat10,19296,0.7220,31.55%
35,ShortLife_cat27,9936,0.3620,31.98%
33,ShortLife_cat23,19944,0.2681,32.38%
3,LongLife_cat17,5184,0.2802,32.71%
24,MediumLife_cat4,259848,0.6748,33.11%
23,MediumLife_cat31,63432,0.5340,33.22%
39,ShortLife_cat4,231408,0.7360,33.27%


1. Linear baseline per shelf‑life segment (DLinear‑style)
This mimics the “simple linear head on top of engineered features” idea: no trees, no deep nonlinearity.

In [20]:
from sklearn.linear_model import Ridge

linear_results = []

for seg in ["ShortLife", "MediumLife", "LongLife"]:
    train_seg = trainsplit_df[trainsplit_df["shelflifebucket"] == seg].copy()
    val_seg   = valsplit_df[valsplit_df["shelflifebucket"] == seg].copy()

    if train_seg.empty or val_seg.empty:
        continue

    X_train = train_seg[final_features].fillna(0)
    y_train = train_seg[target_col].astype(float)

    X_val   = val_seg[final_features].fillna(0)
    y_val   = val_seg[target_col].astype(float)

    # DLinear-style: linear model with L2 regularization
    lin = Ridge(alpha=1.0, random_state=RANDOM_STATE)
    lin.fit(X_train, y_train)

    y_pred = lin.predict(X_val)

    r2_val   = r2_score(y_val, y_pred)
    wape_val = wape(y_val, y_pred)

    print(f"Linear ({seg}) -> R2: {r2_val:.4f} | WAPE: {wape_val:.2f}%")

    linear_results.append({
        "Shelf Life Segment": seg,
        "Algorithm Strategy": "Ridge Linear (DLinear-style)",
        "Eval R2": r2_val,
        "Eval WAPE": wape_val,
    })

linear_results_df = pd.DataFrame(linear_results)
print("\n📦 LINEAR BASELINE RESULTS (Validation):")
display(
    linear_results_df.style.format(
        {"Eval R2": "{:.4f}", "Eval WAPE": "{:.2f}%"}
    )
)

Linear (ShortLife) -> R2: 0.1456 | WAPE: 68.34%
Linear (MediumLife) -> R2: 0.2170 | WAPE: 58.97%
Linear (LongLife) -> R2: 0.5267 | WAPE: 71.28%

📦 LINEAR BASELINE RESULTS (Validation):


,Shelf Life Segment,Algorithm Strategy,Eval R2,Eval WAPE
0,ShortLife,Ridge Linear (DLinear-style),0.1456,68.34%
1,MediumLife,Ridge Linear (DLinear-style),0.2170,58.97%
2,LongLife,Ridge Linear (DLinear-style),0.5267,71.28%


2. Simple “transformer‑style” MLP on tabular features
The paper’s models (TimesNet, iTransformer, SAITS) operate on sequences with attention.

In [21]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class TabularDataset(Dataset):
    def __init__(self, df, feature_cols, target_col):
        self.X = df[feature_cols].fillna(0).values.astype("float32")
        self.y = df[target_col].values.astype("float32")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class TabularMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

In [22]:
def train_mlp_for_segment(seg, train_df, val_df, features, target_col,
                          epochs=5, batch_size=2048, lr=1e-3):
    train_seg = train_df[train_df["shelflifebucket"] == seg].copy()
    val_seg   = val_df[val_df["shelflifebucket"] == seg].copy()

    if train_seg.empty or val_seg.empty:
        print(f"⚠️ Segment {seg}: no data.")
        return None, None

    train_ds = TabularDataset(train_seg, features, target_col)
    val_ds   = TabularDataset(val_seg, features, target_col)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = TabularMLP(input_dim=len(features)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    print(f"\n--- Training MLP for segment: {seg} ---")

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(xb)

        train_loss /= len(train_ds)

        # Simple val loss
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss += loss.item() * len(xb)
        val_loss /= len(val_ds)

        print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    # Evaluate R2 and WAPE on full val set
    model.eval()
    X_val = val_seg[features].fillna(0).values.astype("float32")
    y_val = val_seg[target_col].values.astype("float32")

    with torch.no_grad():
        y_pred = []
        for i in range(0, len(X_val), batch_size):
            xb = torch.from_numpy(X_val[i:i+batch_size]).to(device)
            preds = model(xb).cpu().numpy()
            y_pred.append(preds)
        y_pred = np.concatenate(y_pred)

    r2_val   = r2_score(y_val, y_pred)
    wape_val = wape(y_val, y_pred)

    print(f"MLP ({seg}) -> R2: {r2_val:.4f} | WAPE: {wape_val:.2f}%")

    metrics = {
        "Shelf Life Segment": seg,
        "Algorithm Strategy": "Tabular MLP",
        "Eval R2": r2_val,
        "Eval WAPE": wape_val,
    }

    return model, metrics

In [25]:
mlp_models = {}
mlp_results = []

for seg in ["ShortLife", "MediumLife", "LongLife"]:
    model, metrics = train_mlp_for_segment(
        seg,
        trainsplit_df,
        valsplit_df,
        final_features,
        target_col,
        epochs=10,          # can increase once it works
        batch_size=2048,
        lr=1e-3,
    )
    if model is not None:
        mlp_models[seg] = model
        mlp_results.append(metrics)

mlp_results_df = pd.DataFrame(mlp_results)
print("\n📦 TABULAR MLP RESULTS (Validation):")
display(
    mlp_results_df.style.format(
        {"Eval R2": "{:.4f}", "Eval WAPE": "{:.2f}%"}
    )
)


--- Training MLP for segment: ShortLife ---
Epoch 1: train_loss=1.3341, val_loss=1.4259
Epoch 2: train_loss=0.7811, val_loss=1.2436
Epoch 3: train_loss=0.6733, val_loss=1.1275
Epoch 4: train_loss=0.6319, val_loss=1.0790
Epoch 5: train_loss=0.5979, val_loss=1.1952
Epoch 6: train_loss=0.5809, val_loss=1.1261
Epoch 7: train_loss=0.5666, val_loss=1.1008
Epoch 8: train_loss=0.5552, val_loss=1.1367
Epoch 9: train_loss=0.5470, val_loss=1.3644
Epoch 10: train_loss=0.5448, val_loss=1.5951
MLP (ShortLife) -> R2: 0.2873 | WAPE: 45.63%

--- Training MLP for segment: MediumLife ---
Epoch 1: train_loss=0.7740, val_loss=0.7846
Epoch 2: train_loss=0.5181, val_loss=0.7472
Epoch 3: train_loss=0.4873, val_loss=0.7385
Epoch 4: train_loss=0.4760, val_loss=0.7752
Epoch 5: train_loss=0.4644, val_loss=0.6925
Epoch 6: train_loss=0.4544, val_loss=0.7351
Epoch 7: train_loss=0.4446, val_loss=0.6786
Epoch 8: train_loss=0.4402, val_loss=0.7124
Epoch 9: train_loss=0.4335, val_loss=0.7113
Epoch 10: train_loss=0.4282

,Shelf Life Segment,Algorithm Strategy,Eval R2,Eval WAPE
0,ShortLife,Tabular MLP,0.2873,45.63%
1,MediumLife,Tabular MLP,0.3598,49.04%
2,LongLife,Tabular MLP,0.8992,39.25%


Step 1: compute a simple recovered demand column
Add this to your feature engineering section:

In [26]:
import numpy as np

def compute_recovered_demand(row):
    sale = row["sale_amount"]
    stock_status = np.array(row["hours_stock_status"], dtype=int)

    # Focus on 06–22 (matching stockout_hours logic); indexes 6..22 inclusive
    in_stock_hours = stock_status[6:23]  # 17 hours

    total_hours = len(in_stock_hours)
    hours_in_stock = in_stock_hours.sum()

    # If never stocked or no sale, treat as observed (no inflation)
    if hours_in_stock == 0 or sale == 0:
        return sale

    frac_in_stock = hours_in_stock / total_hours

    # Simple scaling: demand = sale / frac_in_stock
    recovered = sale / frac_in_stock

    # Safety cap to avoid exploding on tiny fractions
    recovered = min(recovered, sale * 3.0)

    return recovered


for df in (trainsplit_df, valsplit_df, eval_df):
    df["latent_demand_recovered"] = df.apply(compute_recovered_demand, axis=1)

In [27]:
target_latent = "latent_demand_recovered"

In [28]:
latent_results = []

for seg in ["ShortLife", "MediumLife", "LongLife"]:
    train_seg = trainsplit_df[trainsplit_df["shelflifebucket"] == seg].copy()
    val_seg   = valsplit_df[valsplit_df["shelflifebucket"] == seg].copy()

    if train_seg.empty or val_seg.empty:
        continue

    X_train = train_seg[final_features].fillna(0)
    y_train = train_seg[target_latent].astype(float)

    X_val   = val_seg[final_features].fillna(0)
    y_val   = val_seg[target_latent].astype(float)

    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_val   = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

    params = {
        "objective": "regression",
        "metric": "l2",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "min_data_in_leaf": 50,
        "verbose": -1,
        "seed": RANDOM_STATE,
    }

    print(f"\n--- LGBM on RECOVERED demand for {seg} ---")
    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=500,
        valid_sets=[lgb_train, lgb_val],
        valid_names=["train", "val"],
        callbacks=[
            early_stopping(stopping_rounds=50, verbose=False),
            log_evaluation(period=0),
        ],
    )

    y_pred = model.predict(X_val, num_iteration=model.best_iteration)

    r2_val   = r2_score(y_val, y_pred)
    wape_val = wape(y_val, y_pred)

    print(f"LGBM_latent ({seg}) -> R2: {r2_val:.4f} | WAPE: {wape_val:.2f}%")

    latent_results.append({
        "Shelf Life Segment": seg,
        "Algorithm Strategy": "LGBM (Recovered Demand)",
        "Eval R2": r2_val,
        "Eval WAPE": wape_val,
    })

latent_results_df = pd.DataFrame(latent_results)
display(
    latent_results_df.style.format(
        {"Eval R2": "{:.4f}", "Eval WAPE": "{:.2f}%"}
    )
)


--- LGBM on RECOVERED demand for ShortLife ---
LGBM_latent (ShortLife) -> R2: 0.7270 | WAPE: 36.83%

--- LGBM on RECOVERED demand for MediumLife ---
LGBM_latent (MediumLife) -> R2: 0.6598 | WAPE: 38.94%

--- LGBM on RECOVERED demand for LongLife ---
LGBM_latent (LongLife) -> R2: 0.8944 | WAPE: 33.43%


,Shelf Life Segment,Algorithm Strategy,Eval R2,Eval WAPE
0,ShortLife,LGBM (Recovered Demand),0.7270,36.83%
1,MediumLife,LGBM (Recovered Demand),0.6598,38.94%
2,LongLife,LGBM (Recovered Demand),0.8944,33.43%


In [29]:
import os, joblib

MODEL_DIR = "/content/drive/MyDrive/Project/Capstone/models"
os.makedirs(MODEL_DIR, exist_ok=True)

final_lgbm_obs = {}
obs_holdout_results = []

segments = ["ShortLife", "MediumLife", "LongLife"]

for seg in segments:
    train_seg = pd.concat([
        trainsplit_df[trainsplit_df["shelflifebucket"] == seg],
        valsplit_df[valsplit_df["shelflifebucket"] == seg]
    ], axis=0)

    eval_seg = eval_df[eval_df["shelflifebucket"] == seg].copy()

    if train_seg.empty or eval_seg.empty:
        print(f"⚠️ Segment {seg}: no data in train+val or eval, skipping.")
        continue

    X_train = train_seg[final_features].fillna(0)
    y_train = train_seg["sale_amount"].astype(float)

    X_eval = eval_seg[final_features].fillna(0)
    y_eval = eval_seg["sale_amount"].astype(float)

    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_eval  = lgb.Dataset(X_eval, label=y_eval, reference=lgb_train)

    params = {
        "objective": "regression",
        "metric": "l2",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "min_data_in_leaf": 50,
        "verbose": -1,
        "seed": RANDOM_STATE,
    }

    print(f"\n--- FINAL LGBM (Observed) Train+Val for {seg} ---")
    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=500,
        valid_sets=[lgb_train, lgb_eval],
        valid_names=["train", "eval"],
        callbacks=[
            early_stopping(stopping_rounds=50, verbose=False),
            log_evaluation(period=0),
        ],
    )

    final_lgbm_obs[seg] = model

    y_pred = model.predict(X_eval, num_iteration=model.best_iteration)
    r2_eval   = r2_score(y_eval, y_pred)
    wape_eval = wape(y_eval, y_pred)

    obs_holdout_results.append({
        "Shelf Life Segment": seg,
        "Target": "sale_amount",
        "Holdout R2": r2_eval,
        "Holdout WAPE": wape_eval,
    })

    # Save model
    model_path = os.path.join(MODEL_DIR, f"lgbm_obs_{seg}.txt")
    model.save_model(model_path)
    print("Saved observed model:", model_path)

obs_holdout_df = pd.DataFrame(obs_holdout_results)
display(obs_holdout_df.style.format({"Holdout R2": "{:.4f}", "Holdout WAPE": "{:.2f}%"}))


--- FINAL LGBM (Observed) Train+Val for ShortLife ---
Saved observed model: /content/drive/MyDrive/Project/Capstone/models/lgbm_obs_ShortLife.txt

--- FINAL LGBM (Observed) Train+Val for MediumLife ---
Saved observed model: /content/drive/MyDrive/Project/Capstone/models/lgbm_obs_MediumLife.txt

--- FINAL LGBM (Observed) Train+Val for LongLife ---
Saved observed model: /content/drive/MyDrive/Project/Capstone/models/lgbm_obs_LongLife.txt


,Shelf Life Segment,Target,Holdout R2,Holdout WAPE
0,ShortLife,sale_amount,0.7355,34.58%
1,MediumLife,sale_amount,0.6041,38.10%
2,LongLife,sale_amount,0.9382,31.59%


In [30]:
final_lgbm_latent = {}
latent_holdout_results = []

for seg in segments:
    train_seg = pd.concat([
        trainsplit_df[trainsplit_df["shelflifebucket"] == seg],
        valsplit_df[valsplit_df["shelflifebucket"] == seg]
    ], axis=0)

    eval_seg = eval_df[eval_df["shelflifebucket"] == seg].copy()

    if train_seg.empty or eval_seg.empty:
        continue

    X_train = train_seg[final_features].fillna(0)
    y_train = train_seg["latent_demand_recovered"].astype(float)

    X_eval = eval_seg[final_features].fillna(0)
    y_eval = eval_seg["latent_demand_recovered"].astype(float)

    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_eval  = lgb.Dataset(X_eval, label=y_eval, reference=lgb_train)

    print(f"\n--- FINAL LGBM (Recovered) Train+Val for {seg} ---")
    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=500,
        valid_sets=[lgb_train, lgb_eval],
        valid_names=["train", "eval"],
        callbacks=[
            early_stopping(stopping_rounds=50, verbose=False),
            log_evaluation(period=0),
        ],
    )

    final_lgbm_latent[seg] = model

    y_pred = model.predict(X_eval, num_iteration=model.best_iteration)
    r2_eval   = r2_score(y_eval, y_pred)
    wape_eval = wape(y_eval, y_pred)

    latent_holdout_results.append({
        "Shelf Life Segment": seg,
        "Target": "latent_demand_recovered",
        "Holdout R2": r2_eval,
        "Holdout WAPE": wape_eval,
    })

    # Save model
    model_path = os.path.join(MODEL_DIR, f"lgbm_latent_{seg}.txt")
    model.save_model(model_path)
    print("Saved recovered model:", model_path)

latent_holdout_df = pd.DataFrame(latent_holdout_results)
display(latent_holdout_df.style.format({"Holdout R2": "{:.4f}", "Holdout WAPE": "{:.2f}%"}))


--- FINAL LGBM (Recovered) Train+Val for ShortLife ---
Saved recovered model: /content/drive/MyDrive/Project/Capstone/models/lgbm_latent_ShortLife.txt

--- FINAL LGBM (Recovered) Train+Val for MediumLife ---
Saved recovered model: /content/drive/MyDrive/Project/Capstone/models/lgbm_latent_MediumLife.txt

--- FINAL LGBM (Recovered) Train+Val for LongLife ---
Saved recovered model: /content/drive/MyDrive/Project/Capstone/models/lgbm_latent_LongLife.txt


,Shelf Life Segment,Target,Holdout R2,Holdout WAPE
0,ShortLife,latent_demand_recovered,0.7339,35.19%
1,MediumLife,latent_demand_recovered,0.6507,38.99%
2,LongLife,latent_demand_recovered,0.9096,33.17%


In [31]:
config = {
    "final_features": final_features,
    "target_obs": "sale_amount",
    "target_latent": "latent_demand_recovered",
}
joblib.dump(config, os.path.join(MODEL_DIR, "model_config.joblib"))
print("Saved model_config.joblib")

Saved model_config.joblib
